# 03 · CLV Model

**Data:** `uci_fact_transactions.parquet` (UCI Online Retail II — ~5.8k customers, ~800k lines)  
**Classical:** BG/NBD + Gamma-Gamma (`lifetimes`)  
**Neural baseline:** MLP regressor on RFM features predicting holdout 90-day revenue

Temporal split: train on orders before cutoff · test on post-cutoff revenue.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)


def save_artifact(name: str, obj) -> Path:
    path = MODELS / name
    joblib.dump(obj, path)
    print(f"Saved → {path.relative_to(PROJECT_ROOT)}")
    return path


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    """Report and drop duplicate rows + rows with NA in required columns (before split)."""
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = out.duplicated(subset=dup_subset, keep="first").sum() if dup_subset else out.duplicated(keep="first").sum()
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum()
    if req:
        out = out.dropna(subset=req)
    print(
        f"[{label}] {n0:,} rows -> {len(out):,} | "
        f"dropped {n_dup:,} duplicates, {na_rows:,} rows with NA"
    )
    if na_rows and (na_by_col > 0).any():
        print("  NA counts:", na_by_col[na_by_col > 0].to_dict())
    return out


In [ ]:
from datetime import timedelta

txn = load_parquet("uci_fact_transactions.parquet")
txn["order_date"] = pd.to_datetime(txn["order_date"])
txn = audit_and_clean(
    txn,
    subset=["customer_id", "order_id", "product_id", "order_date", "quantity", "unit_price"],
    required_cols=["customer_id", "order_id", "order_date", "line_total"],
    label="uci_fact_transactions",
)
print(f"Transactions: {len(txn):,} · Customers: {txn['customer_id'].nunique():,}")
print(f"Span: {txn['order_date'].min().date()} → {txn['order_date'].max().date()}")

# lifetimes format: customer_id, frequency, recency, T, monetary_value
cutoff = txn["order_date"].quantile(0.75)
print(f"Calibration cutoff: {cutoff.date()}")

cal = txn[txn["order_date"] < cutoff].copy()
hold = txn[txn["order_date"] >= cutoff].copy()
hold_end = txn["order_date"].max()

from lifetimes.utils import calibration_and_holdout_data, summary_data_from_transaction_data

summary = summary_data_from_transaction_data(
    txn, "customer_id", "order_date", monetary_value_col="line_total",
    observation_period_end=hold_end,
)
summary.head()


In [ ]:
# Correlation on RFM summary
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(summary[["frequency", "recency", "T", "monetary_value"]].corr(), annot=True, fmt=".2f", ax=ax)
ax.set_title("CLV input correlation (lifetimes summary)")
plt.tight_layout()
plt.show()


In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

bgf = BetaGeoFitter(penalizer_coef=0.1)
bgf.fit(summary["frequency"], summary["recency"], summary["T"])

# Gamma-Gamma needs repeat buyers
gg_data = summary[summary["frequency"] > 0].copy()
ggf = GammaGammaFitter(penalizer_coef=0.1)
ggf.fit(gg_data["frequency"], gg_data["monetary_value"])

t_horizon = 90  # days
summary["p_alive"] = bgf.conditional_probability_alive(
    summary["frequency"], summary["recency"], summary["T"]
)
summary["expected_clv_90d"] = ggf.customer_lifetime_value(
    bgf, summary["frequency"], summary["recency"], summary["T"],
    summary["monetary_value"], time=t_horizon / 30.0,  # lifetimes uses months
)

print("BG/NBD + Gamma-Gamma fitted")
summary[["frequency", "monetary_value", "p_alive", "expected_clv_90d"]].describe().round(2)


In [ ]:
# Holdout validation: actual 90d revenue vs predicted
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

actual = hold.groupby("customer_id")["line_total"].sum().rename("actual_90d")
pred = summary["expected_clv_90d"]
eval_df = pd.concat([pred, actual], axis=1).dropna()
eval_df = eval_df[eval_df.index.isin(actual.index)]

if len(eval_df) > 50:
    mae = mean_absolute_error(eval_df["actual_90d"], eval_df["expected_clv_90d"])
    r2 = r2_score(eval_df["actual_90d"], eval_df["expected_clv_90d"])
    print(f"Probabilistic CLV holdout — MAE: {mae:.2f} · R2: {r2:.3f}")

# Neural baseline on RFM features (drop NA rows before split)
rfm = summary[["frequency", "recency", "T", "monetary_value"]].copy()
rfm["target"] = rfm.index.map(actual).fillna(0)
rfm = audit_and_clean(
    rfm.reset_index(),
    id_col="customer_id",
    required_cols=["customer_id", "frequency", "recency", "T", "monetary_value", "target"],
    label="clv_rfm_summary",
).set_index("customer_id")
X = rfm[["frequency", "recency", "T", "monetary_value"]]
y = rfm["target"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE)),
])
mlp_pipe.fit(X_tr, y_tr)
mlp_pred = mlp_pipe.predict(X_te)
print(f"MLP CLV — MAE: {mean_absolute_error(y_te, mlp_pred):.2f} · R2: {r2_score(y_te, mlp_pred):.3f}")

# Pick best by R2 on test
best_name = "BG/NBD+Gamma-Gamma" if len(eval_df) > 50 and r2 > r2_score(y_te, mlp_pred) else "MLP"
print(f"Selected: {best_name}")


In [ ]:
bgf.save_model(str(MODELS / "03_clv_bgf"))
ggf.save_model(str(MODELS / "03_clv_ggf"))
save_artifact("03_clv_best.joblib", {
    "selected": best_name,
    "bgf_path": "03_clv_bgf",
    "ggf_path": "03_clv_ggf",
    "mlp_pipeline": mlp_pipe,
    "horizon_days": t_horizon,
    "cutoff": str(cutoff.date()),
})
scores_path = MODELS / "03_clv_customer_scores.parquet"
summary.reset_index().to_parquet(scores_path, index=False)
print(f"Saved → {scores_path.relative_to(PROJECT_ROOT)}")
